# Face-Recognition on Google Colab

Runs the same CLI pipeline documented in `CLAUDE.md` (`verify`, `evaluate`, `build-gallery`, `identify`) inside a Colab notebook.

**Before running:** upload your `data/reference/`, `data/test/`, `data/dataset/` images (and, if you already have one, `models/gallery.npz`) into a folder on your Google Drive, e.g. `MyDrive/face-recognition-data/`, mirroring the same subfolder layout. Colab's local disk is wiped every time the runtime disconnects or recycles, so Drive is what makes your data and built gallery persist across sessions.

## 1. Clone the repo

In [ ]:
!git clone https://github.com/ishaans04/Face-Recognition.git
%cd Face-Recognition

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/face-recognition-data'

## 3. Link persisted data/models into the repo layout

`config.py` expects `data/` and `models/` under the repo root, so symlink them straight to Drive. Anything written under `data/` or `models/` (e.g. a freshly built `gallery.npz`) then lands on Drive automatically — no manual copy-back step needed.

In [ ]:
import os

os.makedirs(f'{DRIVE_DATA_DIR}/data/reference', exist_ok=True)
os.makedirs(f'{DRIVE_DATA_DIR}/data/test', exist_ok=True)
os.makedirs(f'{DRIVE_DATA_DIR}/data/dataset', exist_ok=True)
os.makedirs(f'{DRIVE_DATA_DIR}/models', exist_ok=True)

!rm -rf data models
!ln -s "$DRIVE_DATA_DIR/data" data
!ln -s "$DRIVE_DATA_DIR/models" models

## 4. Install dependencies

In [ ]:
!pip install -r requirements.txt

## 5. (Optional) Enable GPU

Only run this cell if you've switched the runtime to a GPU (Runtime -> Change runtime type -> T4 GPU). It installs `onnxruntime-gpu` and flips `config.CTX_ID` from CPU (`-1`) to GPU device `0` for this clone.

In [ ]:
!pip install onnxruntime-gpu
!sed -i 's/^CTX_ID = -1.*/CTX_ID = 0  # GPU device id (set by colab notebook)/' config.py
!grep CTX_ID config.py

## 6. Run the pipeline
Same commands as `CLAUDE.md` — edit paths/flags as needed.

In [ ]:
# 1:1 verification against a reference set
!python main.py verify --reference-dir data/reference/ --test-image data/test/photo.jpg

In [ ]:
# Batch-evaluate labeled pairs (pairs.csv must exist in the repo root)
!python main.py evaluate --pairs-csv pairs.csv

In [ ]:
# Embed data/dataset/<identity>/ into a cached gallery (written to Drive via the models/ symlink)
!python main.py build-gallery --dataset-dir data/dataset/ --output models/gallery.npz

In [ ]:
# 1:N identification against the prebuilt gallery
!python main.py identify --gallery models/gallery.npz --test-image data/test/photo.jpg

## Verifying persistence

Disconnect and reconnect the runtime (Runtime -> Disconnect and delete runtime), then re-run cells 1-3 only. `data/` and `models/gallery.npz` should reappear immediately via the Drive symlink, with no re-upload needed.